# Kronos Backtest â€” Temperature Calibration Sweep (Kaggle)

Walk-forward backtest re-run at several **temperatures** to find which one makes the p10/p90
bands honest. Coverage should land near **80%**; a prior 30-sample run covered only ~6%, so this
uses **100 samples** (production-level) across temperatures `0.6 / 0.9 / 1.2`.

**Secrets** (Add-ons â†’ Secrets): `GH_TOKEN` only. No Turso.

**To run:** Settings â†’ Accelerator â†’ **GPU T4 x2**; for an unattended run use **Save Version â†’
Save & Run All (Commit)**. 100 samples Ã— 3 temperatures is ~3Ã— the work of a single run â€” the
**CONFIG** cell keeps the basket/cadence small so it finishes within a session. Download the
per-temperature CSVs from the **Output** tab when done.

In [ ]:
# Pull the latest pushed code (fresh clone) + the Kronos model repo.
%cd /kaggle/working
from kaggle_secrets import UserSecretsClient
gh = UserSecretsClient().get_secret("GH_TOKEN")
!rm -rf trades && git clone https://{gh}@github.com/tvay11/trades.git
!git clone https://github.com/shiyu-coder/Kronos 2>/dev/null || true
%pip install -q -r trades/forecast/requirements.txt
import os, sys
os.environ['KRONOS_REPO'] = '/kaggle/working/Kronos'
sys.path.insert(0, '/kaggle/working/trades/forecast')
%cd /kaggle/working/trades/forecast

In [ ]:
# ===== CONFIG â€” edit these =====
# Leaner than the magnitude sweep: 3 temperatures x 100 samples is 3x the model calls,
# and the calibration question (does coverage approach 80%?) needs fewer data points.
TICKERS       = ["NVDA", "AAPL", "SPY"]   # widen later once you've seen the runtime
CADENCE       = 42         # ~bimonthly cutoffs (~12) â€” keeps the 3-temp x 100-sample sweep tractable
HORIZONS      = [5, 10, 20, 60]
SAMPLES       = 100        # production-level; wider, more honest bands than the 30-sample run
TEMPERATURES  = [0.6, 0.9, 1.2]
LOOKBACK_DAYS = 1095
OUT_DIR       = "/kaggle/working/backtest_out"

In [ ]:
from backtest import run_backtest

# One sweep per temperature; each writes its own CSVs under OUT_DIR/t<temp>/.
runs = {}
for t in TEMPERATURES:
    print(f"===== temperature {t} =====")
    runs[t] = run_backtest(
        TICKERS, cadence=CADENCE, horizons=HORIZONS, samples=SAMPLES,
        temperature=t, lookback_days=LOOKBACK_DAYS, out_dir=f"{OUT_DIR}/t{t}",
    )

# Calibration headline: band coverage % by temperature (want ~80).
print()
print("COVERAGE % BY TEMPERATURE  (p10/p90 target ~80)")
print("temp   " + "  ".join(f"h{h:>2}" for h in HORIZONS))
for t in TEMPERATURES:
    by_h = {s["horizon"]: s["coveragePct"] for s in runs[t]}
    print(f"{t:<5}  " + "  ".join(f"{by_h.get(h, 0):>5.1f}" for h in HORIZONS))

# Wider bands shouldn't change point accuracy (temperature affects spread, not the median).
print()
print("KRONOS MAPE % BY TEMPERATURE  (lower better; compare to your baselines)")
print("temp   " + "  ".join(f"h{h:>2}" for h in HORIZONS))
for t in TEMPERATURES:
    by_h = {s["horizon"]: s["krMapePct"] for s in runs[t]}
    print(f"{t:<5}  " + "  ".join(f"{by_h.get(h, 0):>5.1f}" for h in HORIZONS))


In [ ]:
# Download each temperature's CSVs (Output tab also has them; everything under /kaggle/working is saved).
from IPython.display import FileLink, display
import os
for t in TEMPERATURES:
    for fname in ("backtest_summary.csv", "backtest_results.csv"):
        p = os.path.join(OUT_DIR, f"t{t}", fname)
        if os.path.exists(p):
            print(f"temperature {t}:")
            display(FileLink(p))